In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# INPUT / OUTPUT
INPUT_CSV = Path("data/MASTER_VARIABLES.csv")
OUTPUT_CSV = Path("data/hazard.csv")

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# 1. DISTRICT-MONTH MEAN
# ---------------------------------------------------------
district_stats = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(district_mean_heatday=("mean_heatday", "mean"))
)

# ---------------------------------------------------------
# 2. MONTHLY Z-SCORE
# ---------------------------------------------------------
district_stats["heat_zscore"] = (
    district_stats.groupby("timeperiod")["district_mean_heatday"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) != 0 else 0)
)

# ---------------------------------------------------------
# 3. BINNING (1–5)
# ---------------------------------------------------------
def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

district_stats["heat_hazard"] = district_stats["heat_zscore"].apply(classify)

# ---------------------------------------------------------
# 4. SAVE ONLY DISTRICT-MONTH OUTPUT (CLEAN)
# ---------------------------------------------------------
district_stats.to_csv(OUTPUT_CSV, index=False)

print(f"Saved clean hazard file: {OUTPUT_CSV}")
print(district_stats.head())


# =============================================================================
# 5. APPEND HAZARD TO MASTER_VARIABLES.CSV
# =============================================================================

master = df.copy()

# drop old column if it exists (prevents _x/_y issues)
if "heat_hazard" in master.columns:
    master = master.drop(columns=["heat_hazard"])

master = master.merge(
    district_stats[
        ["district", "timeperiod", "heat_hazard"]
    ],
    on=["district", "timeperiod"],
    how="left",
)

# save back to master file
master.to_csv(INPUT_CSV, index=False)

print("\nUpdated master file:", INPUT_CSV)
print("Missing hazard values:", master["heat_hazard"].isna().sum())

Saved clean hazard file: data/hazard.csv
  district timeperiod  district_mean_heatday  heat_zscore  heat_hazard
0   Anugul    2023_01               0.000000     0.000000            3
1   Anugul    2023_02               0.000000    -0.445620            3
2   Anugul    2023_03              11.563157     0.064364            3
3   Anugul    2023_04               0.000000    -0.385553            3
4   Anugul    2023_05               5.309366    -0.460448            3

Updated master file: data/MASTER_VARIABLES.csv
Missing hazard values: 0


In [ ]:
# below is for summer hazard 5 years

In [8]:
import pandas as pd
from pathlib import Path

# INPUT / OUTPUT
INPUT_CSV = Path("data/MASTER_VARIABLES.csv")
OUTPUT_CSV = Path("data/hazard_summer.csv")

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# EXTRACT MONTH
# Assumes timeperiod format: YYYY_MM
# ---------------------------------------------------------
df["month"] = (
    df["timeperiod"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .astype(int)
)

# ---------------------------------------------------------
# KEEP ONLY SUMMER MONTHS (March–June)
# ---------------------------------------------------------
summer_df = df[df["month"].isin([3, 4, 5, 6])].copy()

# ---------------------------------------------------------
# DISTRICT-LEVEL SUMMER MEAN
# One value per district for the whole summer season
# ---------------------------------------------------------
district_stats = (
    summer_df.groupby("district", as_index=False)
    .agg(
        summer_mean_heatday=("mean_heatday", "mean")
    )
)

# ---------------------------------------------------------
# SUMMER Z-SCORE ACROSS DISTRICTS
# ---------------------------------------------------------
mean_val = district_stats["summer_mean_heatday"].mean()
std_val = district_stats["summer_mean_heatday"].std(ddof=0)

district_stats["heat_zscore"] = (
    (district_stats["summer_mean_heatday"] - mean_val) / std_val
    if std_val != 0
    else 0
)

# ---------------------------------------------------------
# BINNING (1–5)
# ---------------------------------------------------------
def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

district_stats["heat_hazard"] = district_stats["heat_zscore"].apply(classify)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------
district_stats.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print(district_stats.sort_values("heat_zscore", ascending=False).head())

Saved: data/hazard_summer.csv
          district  summer_mean_heatday  heat_zscore  heat_hazard
25            Puri            25.123999     2.345572            5
11  Jagatsinghapur            25.035156     2.307228            5
6          Cuttack            22.785146     1.336133            4
16      Kendrapara            22.127315     1.052216            4
12         Jajapur            21.722292     0.877410            4
